# WSA_06 — Interpretation & Validation

**Purpose.** Validate invariants and summarize nonlinearity, OOD exposure, and decision sensitivity without causal claims.

> Run from the project repository. Outputs are generated only from the project data and frozen model artifacts.

In [1]:
# Import libraries
from pathlib import Path
import sys, json, pandas as pd, numpy as np, matplotlib.pyplot as plt

In [2]:
# Define config paths
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
CONFIG = PROJECT_ROOT / "configs" / "weather_sensitivity.yaml"
CONFIG

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/configs/weather_sensitivity.yaml')

In [3]:
# Import modules for weather_sensitivity
from src.ontario_peak_risk.weather_sensitivity.common import load_config, ensure_dirs

In [4]:
cfg, project_root = load_config(CONFIG)
paths = ensure_dirs(cfg, project_root)
print("Project root:", project_root)

Project root: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk


In [5]:
r = pd.read_parquet(paths["outputs_dir"] / "WSA_sensitivity_master.parquet")
expected = (
    len(cfg["analysis"]["fsas"])
    * cfg["analysis"]["horizons"]
    * len(cfg["analysis"]["scenarios_c"])
)
assert len(r) == expected, (len(r), expected)
assert (
    r.groupby(["fsa", "horizon"]).size().eq(len(cfg["analysis"]["scenarios_c"])).all()
)
base = r[r.temperature_delta_c.eq(0)]
assert np.allclose(base.forecast_delta_kwh, 0)
assert np.allclose(base.peak_risk_delta, 0)
assert (~base.alert_changed).all()
print("Structural validation: PASS")
print("Domain status:")
print(r.temperature_domain_status.value_counts())
print("Alert changes:", int(r.alert_changed.sum()))

Structural validation: PASS
Domain status:
temperature_domain_status
NORMAL    720
Name: count, dtype: int64
Alert changes: 36


In [6]:
from src.ontario_peak_risk.weather_sensitivity.comparison import (
    summary_by_fsa_scenario,
    symmetry_table,
)

summary = summary_by_fsa_scenario(r)
sym = symmetry_table(r)
summary.to_csv(paths["reports_dir"] / "WSA_06_validation_summary.csv", index=False)
sym.to_csv(paths["reports_dir"] / "WSA_06_nonlinearity_symmetry.csv", index=False)
# Generate a concise report strictly from computed results.
worst = (
    summary.sort_values("max_abs_forecast_delta_kwh", ascending=False).head(1).iloc[0]
)
text = f"""# WSA_06 — Interpretation and Validation

Structural validation: **PASS**.

- Scenario rows: {len(r):,}
- OUT_OF_RANGE rows: {(r.temperature_domain_status == "OUT_OF_RANGE").sum():,}
- CAUTION rows: {(r.temperature_domain_status == "CAUTION").sum():,}
- Peak alert decisions changed vs Baseline: {int(r.alert_changed.sum()):,}
- Largest absolute forecast response observed: {worst.max_abs_forecast_delta_kwh:,.2f} kWh ({worst.fsa}, {worst.scenario}).

## Interpretation boundary
These are frozen-model responses to controlled forecast-origin temperature perturbations. They are not causal weather effects and do not represent target-hour future-weather sensitivity.
"""
(paths["docs_dir"] / "WSA_06_Interpretation_and_Validation.md").write_text(
    text, encoding="utf-8"
)
print(text)
print("WSA_06 RESULT: COMPLETE")

# WSA_06 — Interpretation and Validation

Structural validation: **PASS**.

- Scenario rows: 720
- OUT_OF_RANGE rows: 0
- CAUTION rows: 0
- Peak alert decisions changed vs Baseline: 36
- Largest absolute forecast response observed: 774.13 kWh (M6G, -5C).

## Interpretation boundary
These are frozen-model responses to controlled forecast-origin temperature perturbations. They are not causal weather effects and do not represent target-hour future-weather sensitivity.

WSA_06 RESULT: COMPLETE
